# Plotting Parrots

## Initialising codebase

In [15]:
import sys
import os
from datetime import datetime as dt
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import geopandas as gpd
import plotly.express as px

### Import data

In [16]:

beaufort_scale = pd.read_csv('https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2026/2026-04-14/beaufort_scale.csv')
birds = pd.read_csv('https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2026/2026-04-14/birds.csv')
sea_states = pd.read_csv('https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2026/2026-04-14/sea_states.csv')
ships = pd.read_csv('https://raw.githubusercontent.com/rfordatascience/tidytuesday/main/data/2026/2026-04-14/ships.csv')

/var/folders/8v/b0z9pvdx22ldnktrxjd37zn80000gn/T/ipykernel_78776/3598388994.py:2: DtypeWarning:

Columns (0: sex) have mixed types. Specify dtype option on import or set low_memory=False.



### Check if data imported correctly

In [17]:
beaufort_scale.head()

,wind_speed_class,wind_description,wind_speed_knots_min,wind_speed_knots_max
0,0,calm,0,1.0
1,1,light air,1,3.0
2,2,light breeze,4,6.0
3,3,gentle breeze,7,10.0
4,4,moderate breeze,11,16.0


In [18]:
birds.head()

,bird_observation_id,record_id,species_common_name,species_scientific_name,species_abbreviation,age,wan_plumage_phase,plumage_phase,sex,count,...,sitting_on_ship,in_hand,n_flying_past,flying_past,n_accompanying,accompanying,n_following_ship,following_ship,moulting,naturally_feeding
0,1,1083001,Royal / Wandering albatross,Diomedea epomophora / sanfordi / antipodensis ...,DIOEPOSANANTEXU,NaN,NaN,NaN,NaN,6.0,...,False,False,0.0,False,6.0,True,0.0,False,NaN,False
1,2,1083001,Black-browed albatross sensu lato,Diomedea impavida / melanophrys,DIOIMPMEL,NaN,NaN,NaN,NaN,2.0,...,False,False,0.0,False,2.0,True,0.0,False,NaN,False
2,3,1083001,Cape petrel,Daption capense,DAPCAP,NaN,NaN,NaN,NaN,8.0,...,False,False,0.0,False,8.0,True,0.0,False,NaN,False
3,4,1083001,Fairy prion,Pachyptila turtur,PACTUR,NaN,NaN,NaN,NaN,2.0,...,False,False,0.0,False,2.0,True,0.0,False,NaN,False
4,5,1083001,Sooty shearwater,Puffinus griseus,PUFGRI,NaN,NaN,NaN,NaN,4.0,...,False,False,0.0,False,4.0,True,0.0,False,NaN,False


In [19]:
sea_states.head()

,sea_state_class,sea_state_description,wave_meters_min,wave_meters_max
0,0,"calm, glassy",0.0,0.0
1,1,"calm, rippled",0.0,0.1
2,2,smooth,0.1,0.5
3,3,slight,0.5,1.2
4,4,moderate,1.3,2.5


In [20]:
ships.head()

,record_id,date,time,latitude,longitude,hemisphere,activity,speed,direction,cloud_cover,...,wind_speed_class,wind_direction,air_temperature,pressure,sea_state_class,sea_surface_temperature,depth,observer,census_method,season
0,1083001,1975-10-15,14:00:00,-45.917,165.400,E,"steaming, sailing",15.0,NaN,overcast,...,5.0,NaN,NaN,NaN,5.0,NaN,NaN,D. Jeffcock,full,spring
1,1084001,1975-11-03,13:10:00,-35.533,125.000,E,"steaming, sailing",14.0,NaN,partially cloudy,...,4.0,NaN,NaN,NaN,4.0,NaN,NaN,D. Jeffcock,full,spring
2,1084002,1975-11-04,14:20:00,-37.667,132.250,E,"steaming, sailing",14.5,NaN,overcast,...,4.0,NaN,NaN,NaN,4.0,NaN,NaN,D. Jeffcock,full,spring
3,1084003,1975-11-08,16:15:00,-40.000,162.000,E,"steaming, sailing",14.6,NaN,partially cloudy,...,4.0,NaN,NaN,NaN,4.0,NaN,NaN,D. Jeffcock,full,spring
4,1086001,1975-11-16,12:30:00,-36.167,174.917,E,"steaming, sailing",15.0,NaN,partially cloudy,...,0.0,NaN,NaN,NaN,1.0,NaN,NaN,D. Jeffcock,full,spring


# Plotting by geographical location

In [21]:
def classify_bird(name):
    name = str(name).lower()

    if "storm-petrel" in name or "storm petrel" in name:
        return "Storm petrels"
    elif "shearwater" in name:
        return "Shearwaters"
    elif "albatross" in name:
        return "Albatrosses"
    elif "prion" in name:
        return "Prions"
    elif "petrel" in name:
        return "Petrels"
    elif "fulmar" in name:
        return "Fulmars"
    elif "skua" in name:
        return "Skuas"
    elif "tern" in name:
        return "Terns"
    elif "gull" in name:
        return "Gulls"
    else:
        return "Other"

In [22]:
birds_geo_loc = birds.copy()
birds_geo_loc["group"] = birds["species_common_name"].apply(classify_bird)
birds_geo_loc = birds_geo_loc.groupby(["record_id", "group"]).agg(n_observations= ("group","count"), count = ("count", "sum")).reset_index()
birds_geo_loc.head()

,record_id,group,n_observations,count
0,1083001,Albatrosses,2,8.0
1,1083001,Petrels,1,8.0
2,1083001,Prions,1,2.0
3,1083001,Shearwaters,1,4.0
4,1084001,Albatrosses,2,12.0


In [23]:
top5_groups = birds_geo_loc[birds_geo_loc["group"]!="Other"].groupby("group")["n_observations"].sum().reset_index()
top5_groups = top5_groups.sort_values("n_observations", ascending=False)
top5_groups = top5_groups["group"].head()
print(top5_groups)

0    Albatrosses
3        Petrels
5    Shearwaters
4         Prions
2          Gulls
Name: group, dtype: str


In [24]:
birds_geo_loc = birds_geo_loc[birds_geo_loc["group"].isin(top5_groups)]

In [25]:
geo_loc = ships.copy()[["record_id", "latitude", "longitude"]]
geo_loc.head()

,record_id,latitude,longitude
0,1083001,-45.917,165.400
1,1084001,-35.533,125.000
2,1084002,-37.667,132.250
3,1084003,-40.000,162.000
4,1086001,-36.167,174.917


In [26]:
birds_geo_loc = birds_geo_loc.merge(geo_loc, on="record_id", how="left")
birds_geo_loc.head()

,record_id,group,n_observations,count,latitude,longitude
0,1083001,Albatrosses,2,8.0,-45.917,165.4
1,1083001,Petrels,1,8.0,-45.917,165.4
2,1083001,Prions,1,2.0,-45.917,165.4
3,1083001,Shearwaters,1,4.0,-45.917,165.4
4,1084001,Albatrosses,2,12.0,-35.533,125.0


In [27]:
fig = px.density_map(birds_geo_loc, lat="latitude", lon="longitude", z="n_observations", radius=10, color_continuous_scale="YlOrRd",
    map_style="carto-positron", zoom=2.3, title="Geographical Distribution of Seabirds", animation_frame="group")

fig.update_layout(width=1100, height=650, margin=dict(l=0, r=0, t=50, b=0))

fig.show()